# Wooden Object Segmentation Pipeline

This notebook prepares binary masks, trains a custom U-Net, selects a checkpoint using validation IoU, and evaluates it on the held-out test split. Run cells from top to bottom in Google Colab and update the Google Drive dataset path before loading data. Keep test data out of training and model selection.

In [2]:

# ============================================================
# INSTALLATION AND DEPENDENCIES
# Google Colab
# ============================================================

# General utilities and file handling
!pip install -q requests openpyxl python-dotenv pypdf pyyaml

# Image processing
!pip install -q pillow scikit-image albumentations

# Deep learning evaluation and visualization
!pip install -q scikit-learn matplotlib tqdm

# Optional: advanced segmentation architectures and losses
!pip install -q segmentation-models-pytorch

print("Dependencies installed successfully!")

Dependencies installed successfully!


In [3]:
import torch
import torchvision
import cv2
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import albumentations
import segmentation_models_pytorch as smp

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Albumentations:", albumentations.__version__)
print("Segmentation Models:", smp.__version__)

print("\nAll imports successful!")


PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
CUDA available: False
OpenCV: 4.14.0
NumPy: 2.4.6
Pandas: 2.2.3
Scikit-learn: 1.6.1
Albumentations: 2.0.8
Segmentation Models: 0.6.0

All imports successful!


## 1. Imports and Google Drive Configuration

Import the libraries used by the pipeline, mount Google Drive, and set paths for the dataset, checkpoints, and generated results. Update `DATASET_DIR` to the folder that contains the `train`, `val`, and `test` directories.

## 2. Training Configuration

Set the random seed, image size, batch size, epoch limit, learning rate, and early-stop patience. The device is selected automatically; mixed precision is enabled only on CUDA.

In [9]:

# ============================================================
# 1. IMPORTS AND GOOGLE DRIVE CONFIGURATION
# ============================================================

import os
import json
import random
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# UPDATE THIS PATH TO YOUR ACTUAL GOOGLE DRIVE DATASET FOLDER
# ------------------------------------------------------------

DATASET_DIR = Path(
    "/content/drive/MyDrive/CIP/Dataset"
)

# All outputs will be saved permanently in Google Drive
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Wooden_Object_Dataset_Results/outputs/unet"
)

MODEL_DIR = Path(
    "/content/drive/MyDrive/Wooden_Object_Dataset_Results/models"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. TRAINING CONFIGURATION
# ============================================================

SEED = 42

IMAGE_SIZE = 512
BATCH_SIZE = 2
EPOCHS = 10
LEARNING_RATE = 1e-4
PATIENCE = 10

# Mixed precision is enabled only when CUDA is available
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 50)
print("DEVICE:", DEVICE)
print("CUDA AVAILABLE:", torch.cuda.is_available())
print("MIXED PRECISION:", USE_AMP)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Dataset:", DATASET_DIR)
print("Output:", OUTPUT_DIR)
print("=" * 50)

# Verify dataset directory
if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {DATASET_DIR}"
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEVICE: cpu
CUDA AVAILABLE: False
MIXED PRECISION: False
Dataset: /content/drive/MyDrive/CIP/Dataset
Output: /content/drive/MyDrive/Wooden_Object_Dataset_Results/outputs/unet


## 3. Image Preprocessing

`letterbox` resizes images and masks to a square while preserving aspect ratio. It uses nearest-neighbor interpolation for masks so labels remain discrete.

## 4. Dataset Loader

The dataset class pairs each image with its same-named PNG mask, checks for missing or unmatched files, applies matching flips to image and mask, and converts them to model tensors.

## 5. Create Data Loaders

Build separate training, validation, and test loaders. Augmentation is used only for training; validation and test samples are not shuffled.

In [5]:

# ============================================================
# 3. IMAGE PREPROCESSING
# ============================================================

def letterbox(image, size, is_mask=False):
    """
    Resize while preserving aspect ratio and pad to a square.

    Images use area interpolation.
    Masks use nearest-neighbor interpolation to preserve
    their binary labels.
    """

    h, w = image.shape[:2]

    scale = min(size / w, size / h)

    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    interpolation = (
        cv2.INTER_NEAREST if is_mask else cv2.INTER_AREA
    )

    resized = cv2.resize(
        image,
        (new_w, new_h),
        interpolation=interpolation
    )

    if is_mask:
        canvas = np.zeros((size, size), dtype=image.dtype)
    else:
        canvas = np.zeros(
            (size, size, 3),
            dtype=image.dtype
        )

    x_offset = (size - new_w) // 2
    y_offset = (size - new_h) // 2

    canvas[
        y_offset:y_offset + new_h,
        x_offset:x_offset + new_w
    ] = resized

    return canvas


# ============================================================
# 4. DATASET LOADER
# ============================================================

class WoodenObjectDataset(Dataset):

    def __init__(self, root, split, augment=False):

        self.image_dir = root / split / "images"
        self.mask_dir = root / split / "masks"

        self.augment = augment

        if not self.image_dir.exists():
            raise FileNotFoundError(self.image_dir)

        if not self.mask_dir.exists():
            raise FileNotFoundError(self.mask_dir)

        self.image_paths = sorted([
            p for p in self.image_dir.iterdir()
            if p.suffix.lower() in {
                ".jpg", ".jpeg", ".png"
            }
        ])

        if not self.image_paths:
            raise ValueError(
                f"No images found in {self.image_dir}"
            )

        # Verify image-mask correspondence
        for image_path in self.image_paths:

            mask_path = (
                self.mask_dir / f"{image_path.stem}.png"
            )

            if not mask_path.exists():
                raise FileNotFoundError(
                    f"Missing mask for {image_path.name}: "
                    f"{mask_path}"
                )

        # Detect masks that do not have corresponding images
        image_stems = {
            p.stem for p in self.image_paths
        }

        mask_paths = list(self.mask_dir.glob("*.png"))

        extra_masks = [
            p.name for p in mask_paths
            if p.stem not in image_stems
        ]

        if extra_masks:
            raise ValueError(
                f"Unmatched masks in {split}: {extra_masks[:10]}"
            )

        print(
            f"{split}: {len(self.image_paths)} "
            f"images successfully verified."
        )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):

        image_path = self.image_paths[index]

        mask_path = (
            self.mask_dir / f"{image_path.stem}.png"
        )

        # Read image
        image = cv2.imread(str(image_path))

        if image is None:
            raise ValueError(
                f"Cannot read image: {image_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        # Read grayscale mask
        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE
        )

        if mask is None:
            raise ValueError(
                f"Cannot read mask: {mask_path}"
            )

        if image.shape[:2] != mask.shape[:2]:
            raise ValueError(
                f"Dimension mismatch: {image_path.name}"
            )

        # Threshold grayscale exports so every foreground pixel has the same binary value.
        mask = (mask > 127).astype(np.uint8)

        # Apply identical geometric augmentation
        if self.augment:

            if random.random() > 0.5:
                image = cv2.flip(image, 1)
                mask = cv2.flip(mask, 1)

            if random.random() > 0.5:
                image = cv2.flip(image, 0)
                mask = cv2.flip(mask, 0)

        # Preserve aspect ratio
        image = letterbox(image, IMAGE_SIZE)
        mask = letterbox(
            mask,
            IMAGE_SIZE,
            is_mask=True
        )

        # Normalize image
        image = image.astype(np.float32) / 255.0

        # HWC -> CHW
        image = torch.from_numpy(
            image.transpose(2, 0, 1)
        ).float()

        mask = torch.from_numpy(mask).float()

        return image, mask


# ============================================================
# 5. CREATE DATA LOADERS
# ============================================================

train_dataset = WoodenObjectDataset(
    DATASET_DIR, "train", augment=True
)

val_dataset = WoodenObjectDataset(
    DATASET_DIR, "val", augment=False
)

test_dataset = WoodenObjectDataset(
    DATASET_DIR, "test", augment=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=USE_AMP
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=USE_AMP
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=USE_AMP
)

print("\nDATASET SUMMARY")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))
print(
    "Total:",
    len(train_dataset) +
    len(val_dataset) +
    len(test_dataset)
)


train: 63 images successfully verified.
val: 19 images successfully verified.
test: 9 images successfully verified.

DATASET SUMMARY
Training: 63
Validation: 19
Testing: 9
Total: 91


## 6. U-Net Architecture

Define a binary U-Net with a contracting encoder, a bottleneck, and an expanding decoder. Skip connections combine encoder features with decoder features to retain spatial detail.

## 7. Loss Function

Combine binary cross-entropy on logits with Dice loss to balance pixel-wise classification and foreground-mask overlap. Initialize the model, Adam optimizer, and mixed-precision gradient scaler.

In [6]:

# ============================================================
# 6. U-NET ARCHITECTURE
# ============================================================

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.layers = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.layers(x)


class UNet(nn.Module):

    def __init__(self, num_classes=1):

        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(3, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(256, 512)

        # Decoder
        self.up4 = nn.ConvTranspose2d(
            512, 256, 2, stride=2
        )
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(
            256, 128, 2, stride=2
        )
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(
            128, 64, 2, stride=2
        )
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(
            64, 32, 2, stride=2
        )
        self.dec1 = DoubleConv(64, 32)

        # Binary segmentation output
        self.output = nn.Conv2d(
            32, num_classes, kernel_size=1
        )

    def forward(self, x):

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        # Skip connections restore spatial detail lost during encoder pooling.
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1)


# ============================================================
# 7. LOSS FUNCTION
# ============================================================

class DiceLoss(nn.Module):

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probabilities = torch.sigmoid(logits)

        probabilities = probabilities.reshape(-1)
        targets = targets.reshape(-1)

        intersection = (
            probabilities * targets
        ).sum()

        # Smoothing keeps the ratio defined when prediction and target are both empty.
        dice = (
            2 * intersection + self.smooth
        ) / (
            probabilities.sum() +
            targets.sum() +
            self.smooth
        )

        return 1 - dice


bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()


def combined_loss(logits, masks):

    masks = masks.unsqueeze(1)

    return (
        0.5 * bce_loss(logits, masks)
        +
        0.5 * dice_loss(logits, masks)
    )


# Initialize model
model = UNet(num_classes=1).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# AMP gradient scaler
scaler = torch.cuda.amp.GradScaler(
    enabled=USE_AMP
)

BEST_MODEL_PATH = MODEL_DIR / "unet_best.pth"

print(model)
print(
    "\nTotal parameters:",
    sum(p.numel() for p in model.parameters())
)


UNet(
  (enc1): DoubleConv(
    (layers): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): DoubleConv(
    (layers): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc3): DoubleConv(
    (layers): Sequential(
      (0): Conv2d(64, 128, kernel_size=

/tmp/ipykernel_1585/2280454014.py:165: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(


## 8. Training and Validation

Train for up to the configured epoch limit, then measure validation loss, IoU, precision, and recall. Save the checkpoint with the best validation IoU and stop after the configured patience without improvement. The test split is not used in this loop.

In [10]:

# ============================================================
# 8. TRAINING AND VALIDATION
# ============================================================

best_val_iou = -1.0
epochs_without_improvement = 0

history = []

for epoch in range(EPOCHS):

    # ---------------- TRAINING ----------------

    model.train()

    train_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for images, masks in progress:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        # Mixed precision forward pass
        with torch.cuda.amp.autocast(
            enabled=USE_AMP
        ):

            logits = model(images)

            loss = combined_loss(
                logits,
                masks
            )

        # Scaled backward pass
        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss /= len(train_loader)

    # ---------------- VALIDATION ----------------

    model.eval()

    val_loss = 0.0

    val_tp = 0
    val_fp = 0
    val_fn = 0

    with torch.no_grad():

        for images, masks in val_loader:

            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            with torch.cuda.amp.autocast(
                enabled=USE_AMP
            ):

                logits = model(images)

                loss = combined_loss(
                    logits,
                    masks
                )

            val_loss += loss.item()

            predictions = (
                torch.sigmoid(logits) > 0.5
            )

            targets = masks.unsqueeze(1).bool()

            val_tp += (
                predictions & targets
            ).sum().item()

            val_fp += (
                predictions & ~targets
            ).sum().item()

            val_fn += (
                ~predictions & targets
            ).sum().item()

    val_loss /= len(val_loader)

    val_iou = val_tp / (
        val_tp + val_fp + val_fn + 1e-8
    )

    val_precision = val_tp / (
        val_tp + val_fp + 1e-8
    )

    val_recall = val_tp / (
        val_tp + val_fn + 1e-8
    )

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Val Loss: {val_loss:.4f}"
        f" | Val IoU: {val_iou:.4f}"
        f" | Val Precision: {val_precision:.4f}"
        f" | Val Recall: {val_recall:.4f}"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_iou": val_iou,
        "val_precision": val_precision,
        "val_recall": val_recall
    })

    # Save best model based on validation IoU
    if val_iou > best_val_iou:

        best_val_iou = val_iou

        epochs_without_improvement = 0

        torch.save({

            "model_state_dict": model.state_dict(),
            "epoch": epoch + 1,
            "val_iou": val_iou,
            "seed": SEED,
            "image_size": IMAGE_SIZE

        }, BEST_MODEL_PATH)

        print("[INFO] Best model checkpoint saved.")

    else:

        epochs_without_improvement += 1

    # Early stopping
    if epochs_without_improvement >= PATIENCE:

        print("[INFO] Early stopping triggered.")
        break

    # Save history after every epoch so progress survives
    # even if the Colab session disconnects.
    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "training_history.csv",
        index=False
    )


# Final training history
pd.DataFrame(history).to_csv(
    OUTPUT_DIR / "training_history.csv",
    index=False
)

print("\nTraining completed.")
print("Best validation IoU:", best_val_iou)
print("Best model:", BEST_MODEL_PATH)


Epoch 1/30:   0%|          | 0/32 [00:00<?, ?it/s]

/tmp/ipykernel_1585/1536661964.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_1585/1536661964.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(



Epoch 1/30 | Train Loss: 0.7704 | Val Loss: 0.7590 | Val IoU: 0.0105 | Val Precision: 0.0117 | Val Recall: 0.0990
[INFO] Best model checkpoint saved.


Epoch 2/30:   0%|          | 0/32 [00:00<?, ?it/s]


Epoch 2/30 | Train Loss: 0.7202 | Val Loss: 0.7230 | Val IoU: 0.4111 | Val Precision: 0.4212 | Val Recall: 0.9449
[INFO] Best model checkpoint saved.


Epoch 3/30:   0%|          | 0/32 [00:00<?, ?it/s]

... (30 outputs left out)

Epoch 17/30:   0%|          | 0/32 [00:00<?, ?it/s]


Epoch 17/30 | Train Loss: 0.6184 | Val Loss: 0.6178 | Val IoU: 0.7238 | Val Precision: 0.7441 | Val Recall: 0.9637
[INFO] Best model checkpoint saved.


Epoch 18/30:   0%|          | 0/32 [00:00<?, ?it/s]

RuntimeError: DataLoader worker (pid 38537) is killed by signal: Killed. 

## 9. Load Best Checkpoint

Restore the checkpoint selected by validation IoU before evaluating the model.

## 10. Test Evaluation

Run inference on the held-out test loader, threshold probabilities at 0.5, and save binary prediction masks. Per-image mask IoU and confidence are recorded for later analysis.

## 11. Pixel-Level Metrics

Compute pooled test-set IoU, precision, and recall from pixel-level true positives, false positives, and false negatives. The mean per-image IoU is reported separately.

In [11]:

# ============================================================
# 9. LOAD BEST CHECKPOINT
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    "Loaded best model from epoch:",
    checkpoint["epoch"]
)

print(
    "Best validation IoU:",
    checkpoint["val_iou"]
)


# ============================================================
# 10. TEST EVALUATION
# ============================================================

test_tp = 0
test_fp = 0
test_fn = 0

per_image_ious = []
instance_records = []
image_names = []

prediction_dir = OUTPUT_DIR / "test_predictions"

prediction_dir.mkdir(
    parents=True,
    exist_ok=True
)

with torch.no_grad():

    for batch_index, (images, masks) in enumerate(
        tqdm(test_loader, desc="Testing")
    ):

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        with torch.cuda.amp.autocast(
            enabled=USE_AMP
        ):

            logits = model(images)

        probabilities = torch.sigmoid(logits)

        predictions = probabilities > 0.5

        targets = masks.unsqueeze(1).bool()

        # Pixel-level confusion counts
        test_tp += (
            predictions & targets
        ).sum().item()

        test_fp += (
            predictions & ~targets
        ).sum().item()

        test_fn += (
            ~predictions & targets
        ).sum().item()

        # Process each image individually
        for i in range(images.shape[0]):

            pred = predictions[i, 0].cpu().numpy()

            target = targets[i, 0].cpu().numpy()

            prob = probabilities[i, 0].cpu().numpy()

            # Calculate mask IoU
            intersection = np.logical_and(
                pred, target
            ).sum()

            union = np.logical_or(
                pred, target
            ).sum()

            image_iou = (
                intersection / union
                if union > 0 else 1.0
            )

            per_image_ious.append(
                float(image_iou)
            )

            # Calculate confidence for the predicted mask
            confidence = (
                float(prob[pred].mean())
                if pred.any()
                else float(prob.max())
            )

            # Save predicted binary mask
            prediction_png = (
                pred.astype(np.uint8) * 255
            )

            # Recover original filename from dataset order
            global_index = (
                batch_index * BATCH_SIZE + i
            )

            image_name = (
                test_dataset.image_paths[global_index].name
            )

            image_names.append(image_name)

            output_name = (
                Path(image_name).stem + "_prediction.png"
            )

            cv2.imwrite(
                str(prediction_dir / output_name),
                prediction_png
            )

            # Store only non-empty predicted instances.
            # An empty mask is a missed detection (FN),
            # not a detected object with zero IoU.
            if pred.any():

                instance_records.append({

                    "image_name": image_name,

                    "iou": float(image_iou),

                    "confidence": confidence,

                    "ground_truth_present": bool(
                        target.any()
                    )
                })


# ============================================================
# 11. PIXEL-LEVEL METRICS
# ============================================================

# Pooled counts give dataset-level metrics; per-image IoU is averaged separately.
test_iou = test_tp / (
    test_tp + test_fp + test_fn + 1e-8
)

test_precision = test_tp / (
    test_tp + test_fp + 1e-8
)

test_recall = test_tp / (
    test_tp + test_fn + 1e-8
)

mean_image_iou = float(
    np.mean(per_image_ious)
)

print("\nPIXEL-LEVEL METRICS")
print("IoU:", round(test_iou, 4))
print("Precision:", round(test_precision, 4))
print("Recall:", round(test_recall, 4))
print("Mean Image IoU:", round(mean_image_iou, 4))


Loaded best model from epoch: 17
Best validation IoU: 0.7238299872418426


Testing:   0%|          | 0/5 [00:00<?, ?it/s]

/tmp/ipykernel_1585/421816525.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(



PIXEL-LEVEL METRICS
IoU: 0.7763
Precision: 0.7873
Recall: 0.9825
Mean Image IoU: 0.777


## 12. Instance-Level Mask AP

Rank non-empty predicted masks by confidence and calculate average precision using the documented single-object-per-image assumption. Missed predictions remain false negatives through the total ground-truth count.

## 13. AP at Different IoU Thresholds

Calculate mask AP at IoU thresholds from 0.50 through 0.95 in increments of 0.05, then average those values for mAP@0.5:0.95.

In [12]:

# ============================================================
# 12. INSTANCE-LEVEL MASK AP
# ============================================================

def calculate_instance_ap(
    records,
    total_ground_truth,
    iou_threshold
):
    """
    Calculate mask-based instance AP.

    Assumption:
    One ground-truth wooden object per image.

    Each non-empty predicted mask represents one detection.

    A detection is TP when:
        ground truth exists AND mask IoU >= threshold.

    Otherwise, it is FP.

    Uses 101-point interpolated precision.
    """

    if total_ground_truth == 0:
        return 0.0

    if len(records) == 0:
        return 0.0

    # Sort predictions by confidence, highest first
    sorted_records = sorted(
        records,
        key=lambda x: x["confidence"],
        reverse=True
    )

    true_positives = []
    false_positives = []

    for record in sorted_records:

        is_tp = (
            record["ground_truth_present"]
            and record["iou"] >= iou_threshold
        )

        if is_tp:

            true_positives.append(1)
            false_positives.append(0)

        else:

            true_positives.append(0)
            false_positives.append(1)

    true_positives = np.array(
        true_positives,
        dtype=np.float64
    )

    false_positives = np.array(
        false_positives,
        dtype=np.float64
    )

    # Cumulative TP and FP
    cumulative_tp = np.cumsum(true_positives)
    cumulative_fp = np.cumsum(false_positives)

    # Precision and recall
    precisions = cumulative_tp / (
        cumulative_tp + cumulative_fp + 1e-8
    )

    recalls = cumulative_tp / total_ground_truth

    # Precision envelope
    precisions = np.maximum.accumulate(
        precisions[::-1]
    )[::-1]

    # 101-point interpolated AP
    recall_points = np.linspace(0, 1, 101)

    interpolated_precisions = []

    for recall_point in recall_points:

        valid_precisions = precisions[
            recalls >= recall_point
        ]

        interpolated_precisions.append(
            valid_precisions.max()
            if len(valid_precisions) > 0
            else 0.0
        )

    return float(
        np.mean(interpolated_precisions)
    )


# Count all ground-truth objects, including images
# where the model failed to predict a foreground mask.
total_ground_truth = sum(
    1
    for i in range(len(test_dataset))
    if test_dataset[i][1].sum().item() > 0
)

print("Ground-truth objects:", total_ground_truth)
print("Predicted instances:", len(instance_records))


# ============================================================
# 13. AP AT DIFFERENT IOU THRESHOLDS
# ============================================================

ap_50 = calculate_instance_ap(
    instance_records,
    total_ground_truth,
    0.50
)

iou_thresholds = np.arange(
    0.50,
    0.96,
    0.05
)

ap_values = []

for threshold in iou_thresholds:

    ap = calculate_instance_ap(
        instance_records,
        total_ground_truth,
        threshold
    )

    ap_values.append(ap)

    print(
        f"AP@{threshold:.2f}: {ap:.4f}"
    )

map_50_95 = float(
    np.mean(ap_values)
)

print("\nInstance-level AP@0.5:", round(ap_50, 4))

print(
    "Instance-level mAP@0.5:0.95:",
    round(map_50_95, 4)
)


Ground-truth objects: 9
Predicted instances: 9
AP@0.50: 1.0000
AP@0.55: 1.0000
AP@0.60: 1.0000
AP@0.65: 1.0000
AP@0.70: 1.0000
AP@0.75: 0.6513
AP@0.80: 0.1499
AP@0.85: 0.1188
AP@0.90: 0.0000
AP@0.95: 0.0000

Instance-level AP@0.5: 1.0
Instance-level mAP@0.5:0.95: 0.592


## 14. Final Results

Collect the test metrics and run configuration in one results dictionary. Save the summary, per-image IoU values, AP-by-threshold table, and predicted masks under `OUTPUT_DIR` for reproducibility.

In [13]:

# ============================================================
# 14. FINAL RESULTS
# ============================================================

results = {

    "model": "Custom PyTorch U-Net",

    "task": "Binary Semantic Segmentation",

    "test_pixel_iou": float(test_iou),

    "mean_per_image_mask_iou": float(mean_image_iou),

    "mAP@0.5": float(ap_50),

    "mAP@0.5:0.95": float(map_50_95),

    "precision": float(test_precision),

    "recall": float(test_recall),

    "test_images": len(test_dataset),

    "ground_truth_objects": int(total_ground_truth),

    "best_epoch": int(checkpoint["epoch"]),

    "best_validation_iou": float(checkpoint["val_iou"]),

    "image_size": IMAGE_SIZE,

    "batch_size": BATCH_SIZE,

    "learning_rate": LEARNING_RATE,

    "seed": SEED
}


print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

for metric, value in results.items():

    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")

    else:
        print(f"{metric}: {value}")

print("=" * 60)


# Save evaluation metrics
with open(
    OUTPUT_DIR / "test_metrics.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(results, f, indent=4)


# Save per-image IoU
pd.DataFrame({

    "image_name": image_names,

    "mask_iou": per_image_ious

}).to_csv(
    OUTPUT_DIR / "per_image_test_iou.csv",
    index=False
)


# Save AP at individual IoU thresholds
pd.DataFrame({

    "iou_threshold": iou_thresholds,

    "average_precision": ap_values

}).to_csv(
    OUTPUT_DIR / "instance_ap_results.csv",
    index=False
)


print("\nEvaluation completed successfully.")

print("Results directory:", OUTPUT_DIR)

print("Best model:", BEST_MODEL_PATH)

print("Prediction masks:", prediction_dir)



FINAL TEST RESULTS
model: Custom PyTorch U-Net
task: Binary Semantic Segmentation
test_pixel_iou: 0.7763
mean_per_image_mask_iou: 0.7770
mAP@0.5: 1.0000
mAP@0.5:0.95: 0.5920
precision: 0.7873
recall: 0.9825
test_images: 9
ground_truth_objects: 9
best_epoch: 17
best_validation_iou: 0.7238
image_size: 512
batch_size: 2
learning_rate: 0.0001
seed: 42

Evaluation completed successfully.
Results directory: /content/drive/MyDrive/Wooden_Object_Dataset_Results/outputs/unet
Best model: /content/drive/MyDrive/Wooden_Object_Dataset_Results/models/unet_best.pth
Prediction masks: /content/drive/MyDrive/Wooden_Object_Dataset_Results/outputs/unet/test_predictions
